<a href="https://colab.research.google.com/github/shivani11-glitch/seizure-detection/blob/main/transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import torch
import torch.nn as nn
import math

BATCH_SIZE = 32
SEQ_LEN = 40
EMBED_DIM = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# print(f"Environment initialized. Using: {DEVICE}")

In [16]:
class MultimodalFusion(nn.Module):
    def __init__(self, input_dim=128, d_model=128):
        super().__init__()
        self.projection = nn.Linear(input_dim * 3, d_model)
        self.activation = nn.ELU()
        self.dropout = nn.Dropout(0.1)
        self.bn = nn.BatchNorm1d(SEQ_LEN)

    def forward(self, eeg, ecg, semg):
        combined = torch.cat((eeg, ecg, semg), dim=-1)
        fused = self.projection(combined)
        return self.bn(self.dropout(self.activation(fused)))

# print("Day 1 Fusion Layer Defined.")

In [20]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class SeizureTransformer(nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=2):
        super().__init__()
        self.fusion = MultimodalFusion(input_dim=d_model, d_model=d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=2048,
            dropout=0.1,
            activation="relu",
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        for layer in self.transformer.layers:
            layer.activation = nn.ELU()

    def forward(self, eeg, ecg, semg, mask=None):
        x = self.fusion(eeg, ecg, semg)
        batch_size = x.size(0)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = self.pos_encoder(x)
        output = self.transformer(x, src_key_padding_mask=mask)
        return output[:, 0, :]

# print("Cell 3 Updated: P4 Transformer Engine is now error-free and ready.")

In [22]:
p4_model = SeizureTransformer().to(DEVICE)
teammate_eeg = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM).to(DEVICE)
teammate_ecg = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM).to(DEVICE)
teammate_semg = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM).to(DEVICE)
p4_model.eval()
with torch.no_grad():
    final_summary = p4_model(teammate_eeg, teammate_ecg, teammate_semg)

# print("--- INTEGRATION TEST ---")
# print(f"Received from P2/P3: 3 tensors of shape {teammate_eeg.shape}")
# print(f"Sent to P5: 1 fused summary tensor of shape {final_summary.shape}")
# print("\nSuccess: The P4 module is ready for final integration.")

--- INTEGRATION TEST ---
Received from P2/P3: 3 tensors of shape torch.Size([32, 40, 128])
Sent to P5: 1 fused summary tensor of shape torch.Size([32, 128])

Success: The P4 module is ready for final integration.
